# Project 1: Data Cleaning & Preparation
**DecodeLabs Data Analytics Internship — Industrial Training Kit**

**Analyst:** Data Analyst Intern
**Dataset:** `Dataset_for_Data_Analytics.xlsx` (E-commerce order records)
**Objective:** Inspect, clean, and validate the raw dataset so it is analysis-ready, with zero duplicate IDs, zero incorrectly formatted dates, and all missing values properly handled — per the Project 1 verification gate.

## Workflow
1. Data Inspection
2. Handling Missing Values
3. Removing Duplicates
4. Data Formatting
5. Data Validation
6. Documentation & Export

## 1. Data Inspection

Load the raw dataset and get a first look at its structure, types, and quality.

In [ ]:
import pandas as pd
import numpy as np
import re

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 150)

# Load the raw dataset
df = pd.read_excel('Dataset_for_Data_Analytics.xlsx')
print("Shape:", df.shape)
df.head()

Shape: (1200, 14)


In [ ]:
# Structural overview: column names, dtypes, non-null counts
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   str           
 1   Date             1200 non-null   datetime64[us]
 2   CustomerID       1200 non-null   str           
 3   Product          1200 non-null   str           
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   str           
 7   PaymentMethod    1200 non-null   str           
 8   OrderStatus      1200 non-null   str           
 9   TrackingNumber   1200 non-null   str           
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    str           
 12  ReferralSource   1200 non-null   str           
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[us](1), float64(2), int64(2), str(9)


In [ ]:
# Statistical summary of numeric columns
df.describe()

                      Date     Quantity    UnitPrice  ItemsInCart   TotalPrice
count                 1200  1200.000000  1200.000000  1200.000000  1200.000000
mean   2024-03-22 16:58:48     2.945833   356.412750     5.485000  1053.968300
min    2023-01-01 00:00:00     1.000000    11.390000     1.000000    11.390000
25%    2023-08-03 18:00:00     2.000000   186.062500     4.000000   410.520000
50%    2024-03-23 00:00:00     3.000000   364.210000     5.000000   823.615000
75%    2024-11-08 12:00:00     4.000000   521.570000     7.000000  1578.475000
max    2025-06-30 00:00:00     5.000000   699.930000    10.000000  3456.400000
std                    NaN     1.407557   197.177146     2.281983   819.856558


In [ ]:
# Missing values per column
print("Missing values:")
print(df.isnull().sum())
print()
print("Missing value % :")
print((df.isnull().sum() / len(df) * 100).round(2))

Missing values:
OrderID              0
Date                 0
CustomerID           0
Product              0
Quantity              0
UnitPrice             0
ShippingAddress       0
PaymentMethod         0
OrderStatus           0
TrackingNumber        0
ItemsInCart           0
CouponCode          309
ReferralSource        0
TotalPrice            0
dtype: int64

Missing value % :
OrderID              0.00
Date                 0.00
CustomerID           0.00
Product              0.00
Quantity             0.00
UnitPrice            0.00
ShippingAddress      0.00
PaymentMethod        0.00
OrderStatus          0.00
TrackingNumber       0.00
ItemsInCart          0.00
CouponCode          25.75
ReferralSource       0.00
TotalPrice           0.00
dtype: float64


In [ ]:
# Duplicate checks: exact-row duplicates and duplicate primary keys
print("Full-row duplicates:", df.duplicated().sum())
print("Duplicate OrderID (should be a unique key):", df['OrderID'].duplicated().sum())
print("Duplicate TrackingNumber:", df['TrackingNumber'].duplicated().sum())

# Customers who placed more than one order are expected (repeat customers)
# and are NOT the same thing as a duplicate record - each has a distinct
# OrderID / Date / Product, so we check this separately to avoid a false positive.
repeat_customers = df[df.duplicated('CustomerID', keep=False)]
print("Customers with more than one order (legitimate repeat customers):",
      repeat_customers['CustomerID'].nunique())

Full-row duplicates: 0
Duplicate OrderID (should be a unique key): 0
Duplicate TrackingNumber: 0
Customers with more than one order (legitimate repeat customers): 11


In [ ]:
# Formatting / consistency checks: ID patterns, date validity, categorical spellings
print("OrderID matches pattern ORD#####:",
      df['OrderID'].apply(lambda x: bool(re.fullmatch(r'ORD\d+', str(x)))).all())
print("CustomerID matches pattern C#####:",
      df['CustomerID'].apply(lambda x: bool(re.fullmatch(r'C\d+', str(x)))).all())
print("TrackingNumber matches pattern TRK########:",
      df['TrackingNumber'].apply(lambda x: bool(re.fullmatch(r'TRK\d+', str(x)))).all())
print()
print("Date dtype:", df['Date'].dtype, "| nulls:", df['Date'].isnull().sum())
print()
for col in ['Product', 'PaymentMethod', 'OrderStatus', 'ReferralSource', 'CouponCode']:
    print(col, ":", sorted(df[col].dropna().unique().tolist()))

OrderID matches pattern ORD#####: True
CustomerID matches pattern C#####: True
TrackingNumber matches pattern TRK########: True

Date dtype: datetime64[us] | nulls: 0

Product : ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
PaymentMethod : ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']
OrderStatus : ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']
ReferralSource : ['Email', 'Facebook', 'Google', 'Instagram', 'Referral']
CouponCode : ['FREESHIP', 'SAVE10', 'WINTER15']


In [ ]:
# Business-logic sanity check: does TotalPrice = Quantity * UnitPrice?
calc = (df['Quantity'] * df['UnitPrice']).round(2)
mismatches = (calc - df['TotalPrice']).abs() > 0.01
print("Rows where TotalPrice doesn't match Quantity * UnitPrice:", mismatches.sum())

# Negative / zero sanity checks on numeric fields
print("Quantity <= 0:", (df['Quantity'] <= 0).sum())
print("UnitPrice <= 0:", (df['UnitPrice'] <= 0).sum())
print("TotalPrice <= 0:", (df['TotalPrice'] <= 0).sum())

Rows where TotalPrice doesn't match Quantity * UnitPrice: 0
Quantity <= 0: 0
UnitPrice <= 0: 0
TotalPrice <= 0: 0


### 📋 Inspection Findings

| Check | Result |
|---|---|
| Rows / Columns | 1,200 rows × 14 columns |
| Full-row duplicates | 0 |
| Duplicate `OrderID` / `TrackingNumber` | 0 |
| Missing values | Only `CouponCode` — 309 rows (25.75%) |
| Date column | Already valid `datetime64`, no nulls, range 2023-01-01 → 2025-06-30 |
| ID formats (`OrderID`, `CustomerID`, `TrackingNumber`) | 100% consistent with expected pattern |
| Categorical spellings (Product, PaymentMethod, OrderStatus, ReferralSource) | No casing/typo inconsistencies found |
| `TotalPrice` vs `Quantity × UnitPrice` | Fully consistent, 0 mismatches |
| Negative/zero values in numeric fields | None |

**Conclusion:** the raw dataset is largely well-structured. The **only substantive data-quality issue is missing `CouponCode` values.** The rest of the pipeline still applies the full cleaning/validation workflow (column standardization, duplicate removal, dtype enforcement, text normalization) as a matter of professional practice and to satisfy the Project 1 verification gate.

## 2. Handling Missing Values

`CouponCode` is the only column with nulls (309 / 1200 rows, 25.75%).

**Decision: fill with an explicit placeholder `"No Coupon"` rather than dropping rows or imputing a fake code.**

**Justification:**
- A missing coupon code almost certainly means the customer **checked out without applying any promo code** — this is a legitimate business state, not a data-entry error.
- **Dropping** these 309 rows (Key Requirement option: "Remove rows if minimal impact") is *not* appropriate here — 25.75% is far from minimal, and every dropped row would be a fully valid, complete order.
- **Imputing statistically** (mean/median doesn't apply to a categorical field; filling with the mode, `SAVE10`, would fabricate a discount code the customer never used) would corrupt marketing-attribution analysis.
- A **placeholder category** (`"No Coupon"`) preserves the full order record and makes the "no code used" state explicit and analyzable.

In [ ]:
missing_before = df['CouponCode'].isnull().sum()

df['CouponCode'] = df['CouponCode'].fillna('No Coupon')

print(f"Filled {missing_before} missing CouponCode values with placeholder 'No Coupon'")
print("Remaining missing values in CouponCode:", df['CouponCode'].isnull().sum())

Filled 309 missing CouponCode values with placeholder 'No Coupon'
Remaining missing values in CouponCode: 0


## 3. Removing Duplicates

Inspection already showed **0 full-row duplicates and 0 duplicate `OrderID` values**. We still run an explicit de-duplication step (rather than skip it) so the notebook proves — not just assumes — that the primary key is unique, satisfying the Project 2 gate requirement of a **0% error rate on unique identifiers**.

In [ ]:
rows_before = len(df)

# Drop any exact duplicate rows, and any rows sharing the same OrderID
# (keeping the first occurrence), which is the business primary key.
full_dupes = df.duplicated().sum()
id_dupes = df['OrderID'].duplicated().sum()
print(f"Full-row duplicates found: {full_dupes}")
print(f"Duplicate OrderID found: {id_dupes}")

df = df.drop_duplicates(subset='OrderID', keep='first')
df = df.drop_duplicates(keep='first')

print(f"Rows before: {rows_before}  |  Rows after: {len(df)}")

Full-row duplicates found: 0
Duplicate OrderID found: 0
Rows before: 1200  |  Rows after: 1200


## 4. Data Formatting

- Standardize column names to lowercase `snake_case`.
- Normalize text casing for categorical fields (Title Case) and the coupon field (UPPER, with the placeholder kept readable).
- Strip stray whitespace from all text/ID columns.
- Enforce correct dtypes: `date` → `datetime64`, quantities → nullable integer, prices → rounded float.

In [ ]:
# --- Standardize column names: lowercase, snake_case, no spaces ---
rename_map = {
    'OrderID': 'order_id', 'Date': 'date', 'CustomerID': 'customer_id',
    'Product': 'product', 'Quantity': 'quantity', 'UnitPrice': 'unit_price',
    'ShippingAddress': 'shipping_address', 'PaymentMethod': 'payment_method',
    'OrderStatus': 'order_status', 'TrackingNumber': 'tracking_number',
    'ItemsInCart': 'items_in_cart', 'CouponCode': 'coupon_code',
    'ReferralSource': 'referral_source', 'TotalPrice': 'total_price',
}
df = df.rename(columns=rename_map)
print("Standardized columns:", list(df.columns))

Standardized columns: ['order_id', 'date', 'customer_id', 'product', 'quantity', 'unit_price', 'shipping_address', 'payment_method', 'order_status', 'tracking_number', 'items_in_cart', 'coupon_code', 'referral_source', 'total_price']


In [ ]:
# --- Enforce correct dtypes and normalize text formatting ---

# Dates -> proper datetime (guards against any stray string/mixed formats)
df['date'] = pd.to_datetime(df['date'], errors='coerce')

# Strip whitespace from ID columns
for col in ['order_id', 'customer_id', 'tracking_number']:
    df[col] = df[col].str.strip()

# Title-case categorical text fields for consistent display/grouping
for col in ['product', 'payment_method', 'order_status', 'referral_source']:
    df[col] = df[col].astype(str).str.strip().str.title()

df['shipping_address'] = df['shipping_address'].astype(str).str.strip()

# Coupon codes are business codes -> keep upper case, but preserve the readable placeholder
df['coupon_code'] = df['coupon_code'].astype(str).str.strip().str.upper()
df['coupon_code'] = df['coupon_code'].replace('NO COUPON', 'No Coupon')

# Numeric types: counts as nullable Int64, money rounded to 2 decimals
for col in ['quantity', 'items_in_cart']:
    df[col] = pd.to_numeric(df[col], errors='coerce').astype('Int64')
for col in ['unit_price', 'total_price']:
    df[col] = pd.to_numeric(df[col], errors='coerce').round(2)

print(df.dtypes)

order_id                       str
date                datetime64[us]
customer_id                    str
product                        str
quantity                     Int64
unit_price                 float64
shipping_address               str
payment_method                 str
order_status                   str
tracking_number                str
items_in_cart                Int64
coupon_code                    str
referral_source                str
total_price                float64
dtype: object


In [ ]:
df.head()

## 5. Data Validation

Re-run every check from Step 1 against the cleaned dataset to confirm the Project 1 success criteria are met before export.

In [ ]:
print("1. Missing values per column:")
print(df.isnull().sum())
print()
print("2. Duplicate rows (full):", df.duplicated().sum())
print("   Duplicate order_id:", df['order_id'].duplicated().sum())
print()
print("3. Dtypes:")
print(df.dtypes)

1. Missing values per column:
order_id            0
date                0
customer_id         0
product             0
quantity            0
unit_price          0
shipping_address    0
payment_method      0
order_status        0
tracking_number      0
items_in_cart        0
coupon_code          0
referral_source      0
total_price          0
dtype: int64

2. Duplicate rows (full): 0
   Duplicate order_id: 0

3. Dtypes:
order_id                       str
date                datetime64[us]
customer_id                    str
product                        str
quantity                     Int64
unit_price                 float64
shipping_address               str
payment_method                 str
order_status                   str
tracking_number                str
items_in_cart                Int64
coupon_code                    str
referral_source                str
total_price                float64
dtype: object


In [ ]:
print("4. All dates parsed successfully (0% incorrectly formatted):",
      df['date'].isnull().sum() == 0)
print("   Date range:", df['date'].min(), "to", df['date'].max())
print()
print("5. total_price == quantity * unit_price consistency check:")
calc = (df['quantity'] * df['unit_price']).round(2)
mismatches = (calc - df['total_price']).abs() > 0.01
print("   Mismatches:", mismatches.sum())
print()
print("6. Final categorical value sets:")
for col in ['product','payment_method','order_status','referral_source','coupon_code']:
    print(f"   {col}: {sorted(df[col].unique().tolist())}")
print()
print("7. Final shape:", df.shape)

4. All dates parsed successfully (0% incorrectly formatted): True
   Date range: 2023-01-01 00:00:00 to 2025-06-30 00:00:00

5. total_price == quantity * unit_price consistency check:
   Mismatches: 0

6. Final categorical value sets:
   product: ['Chair', 'Desk', 'Laptop', 'Monitor', 'Phone', 'Printer', 'Tablet']
   payment_method: ['Cash', 'Credit Card', 'Debit Card', 'Gift Card', 'Online']
   order_status: ['Cancelled', 'Delivered', 'Pending', 'Returned', 'Shipped']
   referral_source: ['Email', 'Facebook', 'Google', 'Instagram', 'Referral']
   coupon_code: ['FREESHIP', 'No Coupon', 'SAVE10', 'WINTER15']

7. Final shape: (1200, 14)


✅ **Verification Gate passed:**
- 0% error rate on unique identifiers (`order_id`)
- 0% error rate on date formats
- 0 missing values remaining
- All dtypes correct and consistent

## 6. Export Cleaned Dataset

In [ ]:
df.to_csv('cleaned_dataset.csv', index=False)
df.to_excel('cleaned_dataset.xlsx', index=False)
print("Exported cleaned_dataset.csv and cleaned_dataset.xlsx")

Exported cleaned_dataset.csv and cleaned_dataset.xlsx


## 📄 Summary Report

### Issues Found
| Issue | Column(s) Affected | Severity |
|---|---|---|
| Missing values | `CouponCode` — 309 rows (25.75%) | Moderate — required an explicit decision |
| Non-standard column naming | All columns (PascalCase) | Cosmetic / consistency |
| No duplicate records, IDs, or dates found | — | N/A — dataset was already clean on these dimensions |

### Actions Taken
1. **Inspection** — profiled shape, dtypes, missing values, duplicates, ID/date formats, categorical spellings, and cross-field consistency (`TotalPrice = Quantity × UnitPrice`).
2. **Missing values** — filled 309 missing `CouponCode` entries with the explicit placeholder `"No Coupon"` (representing "no promo code used"), rather than dropping rows or fabricating a code.
3. **Duplicates** — verified and enforced zero full-row duplicates and zero duplicate `OrderID` values.
4. **Formatting** — standardized column names to `snake_case`; title-cased categorical text; trimmed whitespace on ID fields; enforced `datetime64` for dates, nullable integer for counts, and 2-decimal float for currency fields.
5. **Validation** — re-ran every inspection check post-cleaning and confirmed all Project 1 success criteria.
6. **Export** — saved the cleaned dataset as both `cleaned_dataset.csv` and `cleaned_dataset.xlsx`.

### Final Dataset Quality
- **1,200 rows × 14 columns**, no rows dropped.
- **0** missing values.
- **0** duplicate records or IDs.
- **100%** of dates in valid `datetime64` format.
- **0** business-rule mismatches (`total_price` reconciles with `quantity × unit_price` on every row).
- All categorical fields use a single, consistent spelling/casing convention.

**Status: Analysis-ready. Project 1 milestone complete.**